# DA3METRIC-LARGE — KITTI Self-Supervised Fine-tuning & Evaluation

**Unified Pipeline:**

| Phase | Description |
|-------|-------------|
| **1 — Fine-tuning** | Self-supervised training using pre-generated stationary-object masks (no GT depth needed) |
| **2 — Baseline Eval** | Evaluate pre-trained DA3METRIC-LARGE on KITTI Eigen-Split test set |
| **3 — Fine-tuned Eval** | Evaluate fine-tuned checkpoint on same test set |
| **4 — Plots & Summary** | 12 analysis plots × 2 models + 3 comparison plots + summary table |

**Core training signal:** For every stationary object (tree, pole, building…) between consecutive frames,
the depth change `Δ = depth_t − depth_{t+1}` must be *uniform* across all its pixels.
We minimise `Var(Δ)`, `Var(d_{t+1}/d_t)`, MSE toward mean-shift, and edge-aware smoothness.

**Dataset:** KITTI Eigen-Split  
**Masks:** Pre-generated by `inst_seg_notebook.ipynb`, stored at `MASK_ROOT`  
**Model:** `depth-anything/DA3METRIC-LARGE`

---
> **Prerequisites:** `conda activate sam3_da3`  
> `pip install -e Depth-Anything-3/`


In [1]:
"""
da3_kitti_finetune_and_eval.py
══════════════════════════════════════════════════════════════════════════════
Unified pipeline for DA3METRIC-LARGE on KITTI Eigen-Split:

  PHASE 1 – SELF-SUPERVISED FINE-TUNING
  ───────────────────────────────────────
  Fine-tune DA3METRIC-LARGE using pre-generated stationary-object masks
  (from inst_seg_notebook.ipynb). No GT depth labels needed at training time.

  Core training signal: for every stationary object (tree, pole, building …)
  the depth CHANGE between consecutive frames must be UNIFORM across all its
  pixels — because static scene elements move together in the camera view.

  Loss = λ_var·Var(Δdepth)            [uniformity of depth change]
       + λ_ref·MSE(Δ, mean(Δ))        [pull toward mean shift]
       + λ_ratio·Var(d_{t+1}/d_t)     [scale-invariant consistency]
       + λ_smooth·EdgeSmooth(depth)   [within-mask smoothness]
       + λ_kd·SmoothL1(student, self_teacher)  [self-distillation]

  PHASE 2 – BASELINE EVALUATION
  ──────────────────────────────
  Pre-trained DA3METRIC-LARGE inference on KITTI test split → CSV.

  PHASE 3 – FINETUNED EVALUATION
  ─────────────────────────────────
  Fine-tuned checkpoint inference on same test set → second CSV.

  PHASE 4 – COMPARISON & VISUALISATION
  ─────────────────────────────────────
  12 standard analysis plots × 2 models  +  3 comparison plots
  + printed & saved summary tables.

USAGE (command-line)
  python da3_kitti_finetune_and_eval.py --phase all
  python da3_kitti_finetune_and_eval.py --phase train --resume
  python da3_kitti_finetune_and_eval.py --phase eval_baseline
  python da3_kitti_finetune_and_eval.py --phase eval_finetuned
  python da3_kitti_finetune_and_eval.py --phase plots

PREREQUISITES
  conda activate sam3_da3
  git clone https://github.com/ByteDance-Seed/Depth-Anything-3 && cd Depth-Anything-3 && pip install -e .
══════════════════════════════════════════════════════════════════════════════
"""

from __future__ import annotations

# ─── stdlib ──────────────────────────────────────────────────────────────────
import argparse
import csv
import glob
import json
import logging
import math
import os
import sys
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# ─── third-party ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ─── Logging ─────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_pipeline")
logging.getLogger("transformers").setLevel(logging.ERROR)

## §1 · Configuration: CONFIGURATION

In [2]:
@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────────
    # KITTI full dataset (train + test subdirectories)
    kitti_root: str = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small"
    # Pre-generated instance-seg mask root (mirrors KITTI structure)
    mask_root: str  = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output"
    # Where fine-tuned checkpoints are saved
    output_dir: str = "/STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_finetuned"
    # Where evaluation CSVs and plots are saved
    eval_dir: str   = "/STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_eval"
    # Cache for HuggingFace model weights
    cache_dir: str  = "/STORAGE/Arka/cache"

    # ── Model ─────────────────────────────────────────────────────────────────
    model_id: str = "depth-anything/DA3METRIC-LARGE"

    # ── Camera (KITTI default) ─────────────────────────────────────────────────
    focal_length_x: float = 721.5577
    # DA3METRIC internal scale formula: depth = (fx * raw) / metric_scale_denom
    metric_scale_denom: float = 300.0

    # ── Depth clipping ────────────────────────────────────────────────────────
    min_depth: float = 1e-3
    max_depth: float = 80.0

    # ── Mask filtering ────────────────────────────────────────────────────────
    # Only use stationary_object super-category masks for training signal
    train_super_cat: str = "stationary_object"
    # Minimum SAM3 confidence score for evaluation masks
    min_mask_score: float = 0.50
    # Minimum pixel count in combined static mask to use a training pair
    min_mask_pixels: int  = 200

    # ── Dataset (training split) ───────────────────────────────────────────────
    train_split: str = "train"          # sub-folder under kitti_root
    test_split: str  = "test"           # sub-folder under kitti_root
    consecutive_gap: int = 1            # frame index gap (1 = adjacent)
    max_pairs_per_drive: int = 300      # cap pairs per KITTI drive
    val_fraction: float = 0.1           # fraction of drives for validation

    # ── Training ──────────────────────────────────────────────────────────────
    epochs: int          = 20
    lr: float            = 1e-5
    weight_decay: float  = 1e-4
    grad_accum: int      = 8            # effective_batch = grad_accum × batch_size
    batch_size: int      = 1
    grad_clip: float     = 1.0
    mixed_precision: bool = True
    use_8bit_adam: bool  = True
    warmup_epochs: int   = 2
    save_every: int      = 5

    # ── Inference resolution (must be ÷14 for ViT) ────────────────────────────
    infer_h: int = 378                  # 27 × 14
    infer_w: int = 1274                 # 91 × 14  ← nearest to KITTI 1242

    # ── Loss weights ──────────────────────────────────────────────────────────
    lambda_variance:   float = 1.0
    lambda_refinement: float = 0.5
    lambda_ratio:      float = 0.3
    lambda_smooth:     float = 0.05
    lambda_kd:         float = 0.2     # self-distillation (frozen copy → student)

    # ── Resume ────────────────────────────────────────────────────────────────
    resume: bool = False

    # ── Evaluation ────────────────────────────────────────────────────────────
    save_npy: bool       = True
    start_index: int     = 0            # set >0 to resume evaluation
    use_train_for_eval: bool = False    # include train split in evaluation


# ── ImageNet normalisation (matches DA3 preprocessing) ───────────────────────
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_EXTS    = {".jpg", ".jpeg", ".png", ".bmp"}

_da3_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

## §2 · Mask Loading — KITTI-path-aware: MASK LOADING  (KITTI-path-aware)

In [3]:
def get_mask_dir(rgb_path: str, kitti_root: str, mask_root: str) -> str:
    """
    Map a KITTI RGB image path to the corresponding pre-generated mask dir.

    KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
        →
    MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), kitti_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)


def load_masks_for_frame(
    mask_dir: str,
    super_cat_filter: Optional[str] = None,
    min_score: float = 0.0,
) -> List[dict]:
    """
    Load all pre-generated instance masks for one KITTI frame.

    Returns list of dicts with keys:
        uid, local_idx, super_cat, sub_cat, score, box, mask (H×W bool), path
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    # ── Fast path: JSON manifest ──────────────────────────────────────────────
    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)
        instances = []
        for rec in records:
            if super_cat_filter and rec.get("super_cat") != super_cat_filter:
                continue
            if rec.get("score", 1.0) < min_score:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec.get("uid", ""),
                "local_idx": rec.get("local_idx", 0),
                "super_cat": rec.get("super_cat", ""),
                "sub_cat":   rec.get("sub_cat", ""),
                "score":     float(rec.get("score", 1.0)),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # ── Fallback: glob + filename parsing ─────────────────────────────────────
    # Format: <stem>_<super>_<sub>_<idx:04d>_<uid>_<score:.3f>_mask.png
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mask_file.stem[:-5]  # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]
        parts  = suffix.rsplit("_", 3)
        if len(parts) != 4:
            continue
        cat_str, idx_str, uid_str, score_str = parts
        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            continue
        super_cat, sub_cat = cat_parts
        if super_cat_filter and super_cat != super_cat_filter:
            continue
        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            continue
        if score < min_score:
            continue
        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,
            "mask":      mask_np,
            "path":      mask_file,
        })
    instances.sort(key=lambda x: x["local_idx"])
    return instances


def load_combined_static_mask(
    rgb_path: str,
    cfg: Config,
    target_h: int,
    target_w: int,
) -> np.ndarray:
    """
    Merge all stationary_object masks for one KITTI frame into a single
    boolean array of shape (target_h, target_w).
    """
    mask_dir  = get_mask_dir(rgb_path, cfg.kitti_root, cfg.mask_root)
    instances = load_masks_for_frame(
        mask_dir,
        super_cat_filter=cfg.train_super_cat,
        min_score=0.0,
    )
    combined = np.zeros((target_h, target_w), dtype=bool)
    for inst in instances:
        m = inst["mask"]
        if m.shape != (target_h, target_w):
            m = cv2.resize(m.astype(np.uint8), (target_w, target_h),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
        combined |= m
    return combined

## §3 · KITTI Dataset for Fine-tuning: KITTI DATASET FOR FINE-TUNING

In [4]:
class KITTIFramePairDataset(Dataset):
    """
    Consecutive RGB frame pairs from the KITTI TRAIN split.
    Each sample carries pre-loaded static-object masks and KITTI intrinsics.

    Returns dict:
        path_t   : str    (absolute path to frame t)
        path_t1  : str    (absolute path to frame t+1)
        drive    : str    (drive folder name, for grouping)
        mask     : (H, W) bool tensor  [static pixels in BOTH frames]
        n_static : int    [number of True pixels in combined mask]
    """

    def __init__(self, cfg: Config, split: str = "train", seed: int = 42):
        self.cfg   = cfg
        self.split = split
        self.H     = cfg.infer_h
        self.W     = cfg.infer_w
        self.pairs: List[Tuple[str, str, str]] = []  # (path_t, path_t1, drive)
        self._build(seed)

    def _build(self, seed: int):
        split_root = Path(self.cfg.kitti_root) / self.cfg.train_split
        if not split_root.exists():
            log.warning(f"Training split not found: {split_root}")
            return

        rng = np.random.default_rng(seed)
        # Collect all drive dirs: split_root/<date>/<drive>/image_02/data
        date_dirs  = sorted(d for d in split_root.iterdir() if d.is_dir())
        drive_dirs = []
        for date_dir in date_dirs:
            for drive_dir in sorted(date_dir.iterdir()):
                img_dir = drive_dir / "image_02" / "data"
                if img_dir.exists():
                    drive_dirs.append(img_dir)

        # Train / val split on drives
        n_val = max(1, int(len(drive_dirs) * self.cfg.val_fraction))
        perm  = rng.permutation(len(drive_dirs)).tolist()
        val_drives   = {drive_dirs[i] for i in perm[:n_val]}
        train_drives = {drive_dirs[i] for i in perm[n_val:]}
        target_drives = val_drives if self.split == "val" else train_drives

        gap = self.cfg.consecutive_gap
        for img_dir in sorted(target_drives):
            imgs = sorted(
                p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS
            )
            if len(imgs) < gap + 1:
                continue

            drive_pairs = [(str(imgs[i]), str(imgs[i + gap]), img_dir.parent.parent.name)
                           for i in range(len(imgs) - gap)]

            if len(drive_pairs) > self.cfg.max_pairs_per_drive:
                idx = rng.choice(len(drive_pairs),
                                 self.cfg.max_pairs_per_drive, replace=False)
                drive_pairs = [drive_pairs[i] for i in sorted(idx)]

            self.pairs.extend(drive_pairs)

        log.info(f"  {self.split:5s}: {len(self.pairs)} pairs "
                 f"from {len(target_drives)} drives")

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict:
        path_t, path_t1, drive = self.pairs[idx]

        mask_t  = load_combined_static_mask(path_t,  self.cfg, self.H, self.W)
        mask_t1 = load_combined_static_mask(path_t1, self.cfg, self.H, self.W)
        combined = mask_t & mask_t1   # pixels static in BOTH frames

        return {
            "path_t":   path_t,
            "path_t1":  path_t1,
            "drive":    drive,
            "mask":     torch.from_numpy(combined),        # (H, W) bool
            "n_static": int(combined.sum()),
        }

## §4 · Image Preprocessing: IMAGE PREPROCESSING

In [5]:
def load_image_tensor(
    img_path: str,
    target_h: int,
    target_w: int,
    device: torch.device,
) -> torch.Tensor:
    """Load, resize and ImageNet-normalise → float32 tensor (1, 3, H, W)."""
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    t   = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)
    return t.to(device)

## §5 · DA3METRIC-LARGE Forward Pass: DA3METRIC-LARGE FORWARD PASS

In [6]:
def da3_forward(
    model,          # DepthAnything3 instance
    img_tensor: torch.Tensor,   # (1, 3, H, W) float32 on device
    focal_length_x: float,
    metric_scale_denom: float,
    differentiable: bool = True,
) -> torch.Tensor:
    """
    Run DA3METRIC-LARGE forward pass.
    For training (differentiable=True): calls the underlying DepthAnything3Net
    directly to bypass the @torch.inference_mode() decorator on the wrapper.
    For eval (differentiable=False): uses torch.no_grad() as normal.
    """
    inp = img_tensor.unsqueeze(1)  # (1, 1, 3, H, W)

    if differentiable:
        # ── KEY FIX ──────────────────────────────────────────────────────────
        # model is DepthAnything3 (wrapper). model.model is DepthAnything3Net.
        # The wrapper's forward() has @torch.inference_mode() which CANNOT be
        # overridden by torch.enable_grad(). Call the inner net directly.
        # ─────────────────────────────────────────────────────────────────────
        raw_out = model.model(inp, export_feat_layers=[])
    else:
        with torch.no_grad():
            raw_out = model.model(inp, export_feat_layers=[])

    # Unpack output dict/tuple/tensor
    if isinstance(raw_out, dict):
        raw = raw_out.get("depth", raw_out.get("metric_depth", next(iter(raw_out.values()))))
    elif isinstance(raw_out, (tuple, list)):
        raw = raw_out[0]
    else:
        raw = raw_out

    if not isinstance(raw, torch.Tensor):
        raw = torch.as_tensor(np.asarray(raw), device=img_tensor.device).float()

    while raw.dim() < 4:
        raw = raw.unsqueeze(0)
    if raw.shape[1] != 1:
        raw = raw[:, :1]

    depth = (focal_length_x * raw) / metric_scale_denom
    return depth.clamp(min=0.1)


def run_da3_eval_np(
    model,
    image_np: np.ndarray,
    cfg: Config,
    device: torch.device,
) -> np.ndarray:
    """
    Evaluate-mode inference: uint8 RGB (H,W,3) → float32 depth (H, W) in metres.
    Handles resize to DA3-compatible dimensions and upsampling back.
    """
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)

    img_pil = Image.fromarray(image_np).resize((new_w, new_h), Image.BICUBIC)
    t_img   = _da3_transform(img_pil).to(device)
    inp     = t_img.unsqueeze(0).unsqueeze(1)      # (1, 1, 3, H, W)

    with torch.no_grad():
        out = model(inp, export_feat_layers=[])
        if isinstance(out, dict):
            raw = out.get("depth", next(iter(out.values())))
        elif isinstance(out, (tuple, list)):
            raw = out[0]
        else:
            raw = out
        while raw.dim() < 4:
            raw = raw.unsqueeze(0)
        if raw.shape[1] != 1:
            raw = raw[:, :1]
        depth = (cfg.focal_length_x * raw) / cfg.metric_scale_denom
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)

    return depth.squeeze().cpu().numpy().astype(np.float32)

## §6 · Loss Functions: LOSS FUNCTIONS

In [7]:
class TemporalStaticConsistencyLoss(nn.Module):
    """
    Temporal depth consistency for stationary objects between consecutive frames.

    For a truly stationary object (tree, pole, building), the depth of each
    pixel changes by the same amount Δ when the camera translates.
    Therefore Var(Δ) → 0  and  Var(d_{t+1}/d_t) → 0.

    Losses
    ──────
    L₁  variance      : Var(Δ) → 0          Δ = depth_t[mask] − depth_{t+1}[mask]
    L₂  refinement    : MSE(Δ, mean(Δ)) → 0  push each pixel toward mean shift
    L₃  ratio         : Var(d_{t+1}/d_t) → 0 scale-invariant version
    L₄  smoothness    : edge-aware depth smoothness inside mask
    L_kd self-distil  : SmoothL1(student[mask], frozen_self[mask])
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

    def forward(
        self,
        depth_t:    torch.Tensor,           # (1,1,H,W) student depth at frame t
        depth_t1:   torch.Tensor,           # (1,1,H,W) student depth at frame t+1
        mask:       torch.Tensor,           # (H,W) bool
        img_t:      torch.Tensor,           # (1,3,H,W) RGB for smoothness
        teacher_t:  Optional[torch.Tensor] = None,   # (1,1,H,W) frozen
        teacher_t1: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, dict]:

        device = depth_t.device
        d_t    = depth_t [0, 0]   # (H,W)
        d_t1   = depth_t1[0, 0]   # (H,W)

        mask_flat = mask.to(device)
        n_static  = mask_flat.sum()

        if n_static < 4:
            # Use (depth_t * 0).mean() so the dummy is in the comp graph
            # (requires_grad mirrors depth_t). The near-zero check in the
            # training loop will skip backward for this step.
            dummy = (depth_t * 0).mean()
            return dummy, {"var": 0., "refine": 0., "ratio": 0.,
                           "smooth": 0., "kd": 0., "total": 0.}

        st  = d_t [mask_flat]   # (N,)
        st1 = d_t1[mask_flat]

        # L₁ — variance of depth delta
        delta  = st - st1
        l_var  = torch.var(delta)

        # L₂ — refinement toward mean delta (stop-gradient pseudo-label)
        mean_d  = delta.mean().detach()
        l_ref   = F.mse_loss(delta, torch.full_like(delta, mean_d))

        # L₃ — ratio consistency
        ratio   = st1 / st.detach().clamp(min=0.1)
        l_ratio = torch.var(ratio)

        # L₄ — edge-aware smoothness inside mask
        l_smooth = self._edge_smooth(depth_t, img_t, mask_flat)

        # L_kd — self-distillation from frozen copy
        l_kd = depth_t.sum() * 0.0
        if teacher_t is not None and teacher_t1 is not None:
            td_t  = teacher_t [0, 0].to(device)
            td_t1 = teacher_t1[0, 0].to(device)
            l_kd  = (
                F.smooth_l1_loss(st,  td_t [mask_flat].detach(), beta=0.1)
              + F.smooth_l1_loss(st1, td_t1[mask_flat].detach(), beta=0.1)
            ) * 0.5

        cfg   = self.cfg
        total = (cfg.lambda_variance   * l_var
               + cfg.lambda_refinement * l_ref
               + cfg.lambda_ratio      * l_ratio
               + cfg.lambda_smooth     * l_smooth
               + cfg.lambda_kd         * l_kd)

        ld = {k: (v.item() if isinstance(v, torch.Tensor) else v)
              for k, v in {"var": l_var, "refine": l_ref, "ratio": l_ratio,
                           "smooth": l_smooth, "kd": l_kd, "total": total}.items()}
        return total, ld

    @staticmethod
    def _edge_smooth(
        depth: torch.Tensor,   # (1,1,H,W)
        image: torch.Tensor,   # (1,3,H,W)
        mask:  torch.Tensor,   # (H,W) bool
    ) -> torch.Tensor:
        mean_d = depth.mean().clamp(min=1e-3)
        d_n    = depth / mean_d

        gd_x   = (d_n[:, :, :, 1:] - d_n[:, :, :, :-1]).abs()
        gd_y   = (d_n[:, :, 1:, :] - d_n[:, :, :-1, :]).abs()
        gray   = image.mean(dim=1, keepdim=True)
        gi_x   = (gray[:, :, :, 1:] - gray[:, :, :, :-1]).abs()
        gi_y   = (gray[:, :, 1:, :] - gray[:, :, :-1, :]).abs()
        wx, wy = torch.exp(-gi_x), torch.exp(-gi_y)

        m  = mask.unsqueeze(0).unsqueeze(0).float()
        mx = m[:, :, :, 1:];  my = m[:, :, 1:, :]
        nx = mx.sum().clamp(min=1);  ny = my.sum().clamp(min=1)
        return (gd_x * wx * mx).sum() / nx + (gd_y * wy * my).sum() / ny

## §7 · Model Loading & Optimizer: MODEL LOADING & OPTIMIZER

In [8]:
def load_da3_model(
    model_id: str,
    cache_dir: str,
    device: torch.device,
    train: bool = False,
) -> "DepthAnything3":
    try:
        from depth_anything_3.api import DepthAnything3
    except ImportError:
        sys.exit("❌  depth_anything_3 not installed. Run: pip install -e Depth-Anything-3/")
    log.info(f"Loading {model_id} …")
    mdl = DepthAnything3.from_pretrained(model_id, cache_dir=cache_dir).to(device)
    mdl.train(train)
    log.info(f"  Ready on {device}  |  mode={'train' if train else 'eval'}")
    return mdl


def build_optimizer(params, cfg: Config) -> torch.optim.Optimizer:
    if cfg.use_8bit_adam:
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, lr=cfg.lr,
                                       weight_decay=cfg.weight_decay)
            log.info("Using 8-bit AdamW (bitsandbytes).")
            return opt
        except ImportError:
            log.warning("bitsandbytes not found → using standard AdamW.")
    return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)


def build_scheduler(opt, cfg: Config, steps_per_epoch: int):
    total  = cfg.epochs * steps_per_epoch
    warmup = cfg.warmup_epochs * steps_per_epoch
    sch_w  = torch.optim.lr_scheduler.LinearLR(
        opt, start_factor=0.01, end_factor=1.0, total_iters=warmup)
    sch_c  = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(total - warmup, 1), eta_min=cfg.lr * 0.01)
    return torch.optim.lr_scheduler.SequentialLR(
        opt, [sch_w, sch_c], milestones=[warmup])

## §8 · Checkpoint Utilities: CHECKPOINT UTILITIES

In [9]:
def save_checkpoint(model, optimizer, epoch: int, val_loss: float,
                    cfg: Config, name: str) -> Path:
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    path = out / name
    # Save only trainable params to keep file small
    sd = {k: v for k, v in model.state_dict().items()
          if any(k == n for n, p in model.named_parameters() if p.requires_grad)}
    if not sd:
        sd = model.state_dict()
    torch.save({"epoch": epoch, "val_loss": val_loss,
                "model_id": cfg.model_id,
                "model_state": sd,
                "optimizer": optimizer.state_dict()}, path)
    log.info(f"  ✔  Checkpoint → {path}")
    return path


def load_checkpoint(model, optimizer, cfg: Config) -> int:
    """Load latest epoch_*.pt from output_dir. Returns last epoch."""
    ckpts = sorted(Path(cfg.output_dir).glob("epoch_*.pt"))
    if not ckpts:
        log.info("No checkpoint found — starting from scratch.")
        return 0
    ckpt = torch.load(ckpts[-1], map_location="cpu")
    missing, unexpected = model.load_state_dict(ckpt["model_state"], strict=False)
    try:
        optimizer.load_state_dict(ckpt["optimizer"])
    except Exception:
        pass
    epoch = ckpt.get("epoch", 0)
    log.info(f"Resumed from {ckpts[-1].name}  (epoch {epoch})")
    if missing:
        log.warning(f"  Missing keys (first 5): {missing[:5]}")
    return epoch


def load_best_checkpoint(model, cfg: Config) -> bool:
    """Load best_model.pt for evaluation. Returns True on success."""
    path = Path(cfg.output_dir) / "best_model.pt"
    if not path.exists():
        log.warning(f"No best_model.pt at {path}")
        # Try latest epoch checkpoint
        ckpts = sorted(Path(cfg.output_dir).glob("epoch_*.pt"))
        if ckpts:
            path = ckpts[-1]
            log.info(f"  Falling back to latest checkpoint: {path.name}")
        else:
            return False
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"], strict=False)
    log.info(f"Loaded fine-tuned weights from {path.name}  "
             f"(val_loss={ckpt.get('val_loss', '?'):.4f})")
    return True

## §9 · Training Loop: TRAINING LOOP

In [10]:
def train(cfg: Config) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not torch.cuda.is_available():
        log.warning("CUDA not available — training on CPU will be very slow.")
    else:
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        log.info(f"GPU: {torch.cuda.get_device_name(0)}  ({mem:.1f} GB)")

    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

    # ── Datasets ──────────────────────────────────────────────────────────────
    train_ds = KITTIFramePairDataset(cfg, split="train")
    val_ds   = KITTIFramePairDataset(cfg, split="val")

    def collate(batch):
        keys = batch[0].keys()
        out  = {}
        for k in keys:
            if isinstance(batch[0][k], str):
                out[k] = [b[k] for b in batch]
            elif isinstance(batch[0][k], torch.Tensor):
                out[k] = torch.stack([b[k] for b in batch])
            else:
                out[k] = [b[k] for b in batch]
        return out

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              shuffle=True, num_workers=2, pin_memory=True,
                              persistent_workers=True, collate_fn=collate)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                              num_workers=2, collate_fn=collate)

    # ── Model (student + frozen self-teacher for KD) ───────────────────────────
    student = load_da3_model(cfg.model_id, cfg.cache_dir, device, train=True)
    student.model.train()   # explicitly set inner net to train mode
    teacher = load_da3_model(cfg.model_id, cfg.cache_dir, device, train=False)
    student.model.eval()    # already set, but explicit is better
    for p in teacher.parameters():
        p.requires_grad_(False)

    trainable = [p for p in student.parameters() if p.requires_grad]
    log.info(f"Trainable params: {sum(p.numel() for p in trainable)/1e6:.1f}M")

    optimizer  = build_optimizer(trainable, cfg)
    steps_pe   = max(len(train_loader) // cfg.grad_accum, 1)
    scheduler  = build_scheduler(optimizer, cfg, steps_pe)
    scaler     = GradScaler("cuda", enabled=cfg.mixed_precision)
    loss_fn    = TemporalStaticConsistencyLoss(cfg)

    start_epoch = 0
    if cfg.resume:
        start_epoch = load_checkpoint(student, optimizer, cfg)

    # NOTE: Do NOT reassign optimizer.param_groups[0]["params"] here —
    # it corrupts optimizer state (momentum buffers etc.) loaded from checkpoint.
    # The optimizer was already built from `trainable` which equals the
    # requires_grad parameters, so no fixup is needed.

    best_val    = float("inf")
    global_step = start_epoch * steps_pe

    # TensorBoard (optional)
    writer = None
    try:
        from torch.utils.tensorboard import SummaryWriter
        writer = SummaryWriter(log_dir=str(Path(cfg.output_dir) / "runs"))
        log.info(f"TensorBoard: tensorboard --logdir {Path(cfg.output_dir)/'runs'}")
    except ImportError:
        pass

    log.info(f"\n{'═'*65}")
    log.info(f"  Epochs: {cfg.epochs}  |  Train pairs: {len(train_ds)}  |  "
             f"Val pairs: {len(val_ds)}")
    log.info(f"  Effective batch: {cfg.batch_size * cfg.grad_accum}")
    log.info(f"{'═'*65}\n")

    for epoch in range(start_epoch + 1, cfg.epochs + 1):
        student.train()
        meters: dict = defaultdict(list)
        t0 = time.time()
        optimizer.zero_grad()

        for step_i, batch in enumerate(train_loader):
            path_t   = batch["path_t"][0]
            path_t1  = batch["path_t1"][0]
            mask     = batch["mask"][0]       # (H,W) bool
            n_static = batch["n_static"][0] if isinstance(batch["n_static"][0], int) \
                       else batch["n_static"][0].item()

            if n_static < cfg.min_mask_pixels:
                continue

            try:
                img_t_tensor  = load_image_tensor(path_t,  cfg.infer_h, cfg.infer_w, device)
                img_t1_tensor = load_image_tensor(path_t1, cfg.infer_h, cfg.infer_w, device)

                # Self-teacher inference (frozen) — must run BEFORE student forward
                # so no_grad doesn't pollute the student's autocast graph
                with torch.no_grad():
                    teach_t  = da3_forward(teacher, img_t_tensor,
                                           cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)
                    teach_t1 = da3_forward(teacher, img_t1_tensor,
                                           cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)

                # FIX: torch.enable_grad() explicitly re-enables grad inside
                # autocast. Without this, the outer torch.no_grad() teacher
                # block could leave grad disabled if set_grad_enabled is absent.
                with torch.enable_grad(), autocast("cuda", enabled=cfg.mixed_precision):
                    depth_t  = da3_forward(student, img_t_tensor,
                                           cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=True)
                    depth_t1 = da3_forward(student, img_t1_tensor,
                                           cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=True)
                    loss, ld = loss_fn(depth_t, depth_t1,
                                       mask, img_t_tensor.detach(),
                                       teach_t, teach_t1)

            except Exception as e:
                log.warning(f"Step {step_i} failed: {e}")
                optimizer.zero_grad()
                continue

            # Skip non-finite or near-zero losses (e.g. early-return dummy)
            if not torch.isfinite(loss) or abs(loss.item()) < 1e-12:
                continue

            scaler.scale(loss / cfg.grad_accum).backward()

            for k, v in ld.items():
                meters[k].append(v)

            if (step_i + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(trainable, cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 20 == 0:
                    lr  = optimizer.param_groups[0]["lr"]
                    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.
                    avg = {k: np.mean(vs[-20:]) for k, vs in meters.items() if vs}
                    log.info(
                        f"Ep{epoch:03d} step{global_step:06d}  "
                        f"loss={avg.get('total',0):.4f}  "
                        f"var={avg.get('var',0):.4f}  "
                        f"refine={avg.get('refine',0):.4f}  "
                        f"ratio={avg.get('ratio',0):.4f}  "
                        f"kd={avg.get('kd',0):.4f}  "
                        f"lr={lr:.2e}  mem={mem:.2f}GB"
                    )
                    if writer:
                        for k, v in avg.items():
                            writer.add_scalar(f"train/{k}", v, global_step)
                        writer.add_scalar("train/lr", lr, global_step)

        epoch_avg = {k: float(np.mean(vs)) for k, vs in meters.items() if vs}
        log.info(f"\n  Epoch {epoch}/{cfg.epochs}  "
                 f"train_loss={epoch_avg.get('total',0):.4f}  "
                 f"({time.time()-t0:.0f}s)")

        # ── Validation ────────────────────────────────────────────────────────
        if epoch % max(1, cfg.save_every // 2) == 0:
            student.eval()
            val_losses = []
            with torch.no_grad():
                for batch in val_loader:
                    path_t   = batch["path_t"][0]
                    path_t1  = batch["path_t1"][0]
                    mask     = batch["mask"][0]
                    n_static = batch["n_static"][0].item() \
                               if hasattr(batch["n_static"][0], "item") \
                               else batch["n_static"][0]
                    if n_static < cfg.min_mask_pixels:
                        continue
                    try:
                        img_t  = load_image_tensor(path_t,  cfg.infer_h, cfg.infer_w, device)
                        img_t1 = load_image_tensor(path_t1, cfg.infer_h, cfg.infer_w, device)
                        dt   = da3_forward(student, img_t,  cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)
                        dt1  = da3_forward(student, img_t1, cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)
                        tt   = da3_forward(teacher, img_t,  cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)
                        tt1  = da3_forward(teacher, img_t1, cfg.focal_length_x,
                                           cfg.metric_scale_denom, differentiable=False)
                        _, ld = loss_fn(dt, dt1, mask, img_t.detach(), tt, tt1)
                        if ld.get("total", 0) > 0:
                            val_losses.append(ld["total"])
                    except Exception:
                        pass

            val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
            log.info(f"  Val loss: {val_loss:.4f}")
            if writer:
                writer.add_scalar("val/total", val_loss, epoch)

            if val_loss < best_val:
                best_val = val_loss
                save_checkpoint(student, optimizer, epoch, val_loss,
                                cfg, "best_model.pt")
                log.info(f"  ★ New best val loss: {best_val:.4f}")

        # ── Periodic checkpoint ───────────────────────────────────────────────
        if epoch % cfg.save_every == 0:
            save_checkpoint(student, optimizer, epoch,
                            epoch_avg.get("total", 0.0),
                            cfg, f"epoch_{epoch:04d}.pt")

    log.info(f"\nTraining complete. Best val loss: {best_val:.4f}")
    if writer:
        writer.close()

## §10 · KITTI Indexing for Evaluation: KITTI INDEXING

In [11]:
def index_kitti(cfg: Config) -> pd.DataFrame:
    """
    Walk the KITTI Eigen-Split test (and optionally train) directories.
    Returns a DataFrame with columns: folder, sequence, drive, filename,
    gt_path, rgb_path.
    """
    rows = []

    def _collect(split_root: str, tag: str):
        pattern  = os.path.join(split_root, "*", "*_sync",
                                "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive,
                                    "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(folder=tag, sequence=date, drive=drive,
                             filename=fname, gt_path=gt_path, rgb_path=rgb_path))

    _collect(os.path.join(cfg.kitti_root, cfg.test_split),  "test")
    if cfg.use_train_for_eval:
        _collect(os.path.join(cfg.kitti_root, cfg.train_split), "train")

    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info(f"Dataset indexed: {len(df)} frames, {df['drive'].nunique()} drives.")
    return df

## §11 · Metric & Analysis Helpers: METRIC & ANALYSIS HELPERS

In [12]:
def compute_depth_errors(gt: np.ndarray,
                          pred: np.ndarray,
                          mask: np.ndarray) -> dict:
    gt_v    = gt[mask]
    pred_v  = np.maximum(pred[mask], 1e-8)
    thresh  = np.maximum(gt_v / pred_v, pred_v / gt_v)
    return dict(
        rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2))),
        abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v)),
        sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v)),
        rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2))),
        delta1   = float(np.mean(thresh < 1.25)),
        delta2   = float(np.mean(thresh < 1.25 ** 2)),
        delta3   = float(np.mean(thresh < 1.25 ** 3)),
    )


def eigen_crop_mask(shape) -> np.ndarray:
    h, w = shape[:2]
    m    = np.zeros((h, w), dtype=bool)
    m[44:min(396, h), :] = True
    return m


def compute_all_scales(gt: np.ndarray, pred_raw: np.ndarray,
                        mask: np.ndarray, cfg: Config) -> dict:
    gt_v   = gt[mask]
    pred_c = np.clip(pred_raw, cfg.min_depth, cfg.max_depth)
    res    = {}

    # Unscaled
    m0 = compute_depth_errors(gt, pred_c, mask)
    res["unscaled"] = {"scale": 1., "shift": 0., "depth_map": pred_c, **m0,
                        "avg_depth": float(np.mean(pred_c[mask])),
                        "min_depth": float(np.min(pred_c[mask]))}

    # Median
    s_med   = float(np.median(gt_v) / np.median(pred_c[mask]))
    pred_m  = np.clip(pred_c * s_med, cfg.min_depth, cfg.max_depth)
    m1      = compute_depth_errors(gt, pred_m, mask)
    res["median"] = {"scale": s_med, "shift": 0., "depth_map": pred_m, **m1,
                      "avg_depth": float(np.mean(pred_m[mask])),
                      "min_depth": float(np.min(pred_m[mask]))}

    # Mean
    s_mean  = float(np.mean(gt_v) / np.mean(pred_c[mask]))
    pred_mn = np.clip(pred_c * s_mean, cfg.min_depth, cfg.max_depth)
    m2      = compute_depth_errors(gt, pred_mn, mask)
    res["mean"] = {"scale": s_mean, "shift": 0., "depth_map": pred_mn, **m2,
                    "avg_depth": float(np.mean(pred_mn[mask])),
                    "min_depth": float(np.min(pred_mn[mask]))}

    # Affine OLS
    pf   = pred_c[mask].ravel()
    A    = np.vstack([pf, np.ones_like(pf)]).T
    p, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)[:2]
    s_af, sh_af = float(p[0]), float(p[1])
    pred_af     = np.clip(s_af * pred_c + sh_af, cfg.min_depth, cfg.max_depth)
    m3 = compute_depth_errors(gt, pred_af, mask)
    res["affine"] = {"scale": s_af, "shift": sh_af, "depth_map": pred_af, **m3,
                      "avg_depth": float(np.mean(pred_af[mask])),
                      "min_depth": float(np.min(pred_af[mask]))}
    return res


CSV_COLUMNS = [
    "img_id", "file_name", "mask_id", "mask_uid", "mask_score",
    "super_cat", "sub_cat", "horizontal_angle",
    "unscaled_rmse", "unscaled_abs_rel", "unscaled_sq_rel", "unscaled_rmse_log",
    "unscaled_delta1", "unscaled_delta2", "unscaled_delta3",
    "unscaled_avg_depth", "unscaled_min_depth",
    "median_scale",
    "median_rmse", "median_abs_rel", "median_sq_rel", "median_rmse_log",
    "median_delta1", "median_delta2", "median_delta3",
    "median_avg_depth", "median_min_depth",
    "mean_scale",
    "mean_rmse", "mean_abs_rel", "mean_sq_rel", "mean_rmse_log",
    "mean_delta1", "mean_delta2", "mean_delta3",
    "mean_avg_depth", "mean_min_depth",
    "affine_scale", "affine_shift",
    "affine_rmse", "affine_abs_rel", "affine_sq_rel", "affine_rmse_log",
    "affine_delta1", "affine_delta2", "affine_delta3",
    "affine_avg_depth", "affine_min_depth",
    "lidar_avg_depth", "lidar_min_depth",
    "rgb_image_path", "gt_depth_map_path", "da3_pred_map_path", "mask_path",
]


def process_dataset_eval(
    model,
    val_list: pd.DataFrame,
    cfg: Config,
    output_csv: str,
    out_dir: str,
    label: str = "model",
) -> None:
    """
    Run depth inference + per-mask metric computation for all frames in val_list.
    Appends rows to output_csv (resumable via START_INDEX equivalent).
    """
    device = next(model.parameters()).device

    RGB_DIR  = os.path.join(out_dir, "rgb_images"); os.makedirs(RGB_DIR,  exist_ok=True)
    GT_DIR   = os.path.join(out_dir, "gt_depth");   os.makedirs(GT_DIR,   exist_ok=True)
    DA3_DIR  = os.path.join(out_dir, "da3_pred");   os.makedirs(DA3_DIR,  exist_ok=True)
    MASK_OUT = os.path.join(out_dir, "masks");       os.makedirs(MASK_OUT, exist_ok=True)

    # Resume: figure out which img_ids are already done
    done_ids: set = set()
    if os.path.exists(output_csv):
        try:
            done_df  = pd.read_csv(output_csv, usecols=["img_id"])
            done_ids = set(done_df["img_id"].unique())
        except Exception:
            pass
    else:
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(output_csv, index=False)

    skipped_no_masks = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list),
                            desc=f"Eval [{label}]"):
        if index in done_ids:
            continue

        base_fn = (os.path.splitext(row["folder"])[0]
                   + "_" + row["drive"]
                   + "_" + os.path.splitext(row["filename"])[0])

        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = np.array(Image.open(row["gt_path"])).astype(np.uint16) / 256.0
            H, W, _  = image_np.shape
            cx        = W / 2.0

            # Load all masks (not just stationary — for full evaluation)
            mask_dir   = get_mask_dir(row["rgb_path"], cfg.kitti_root, cfg.mask_root)
            all_inst   = load_masks_for_frame(mask_dir)

            if not all_inst:
                skipped_no_masks += 1
                continue

            pred_raw = run_da3_eval_np(model, image_np, cfg, device)

            rgb_out  = os.path.join(RGB_DIR,  f"{base_fn}.png")
            gt_out   = os.path.join(GT_DIR,   f"{base_fn}.npy")
            da3_out  = os.path.join(DA3_DIR,  f"{base_fn}.npy")
            Image.fromarray(image_np).save(rgb_out)
            if cfg.save_npy:
                np.save(gt_out,  gt_depth)
                np.save(da3_out, pred_raw)

            crop = eigen_crop_mask(gt_depth.shape)

            for inst in all_inst:
                seg   = inst["mask"]
                score = inst["score"]
                if score < cfg.min_mask_score:
                    continue
                if seg.shape != (H, W):
                    seg = cv2.resize(seg.astype(np.uint8), (W, H),
                                     interpolation=cv2.INTER_NEAREST).astype(bool)
                final = (seg & (gt_depth > cfg.min_depth)
                             & (gt_depth < cfg.max_depth)
                             & crop)
                if not np.any(final):
                    continue

                scales     = compute_all_scales(gt_depth, pred_raw, final, cfg)
                lidar_v    = gt_depth[final]
                _, xc      = np.where(seg)
                angle_deg  = math.degrees(
                    math.atan((float(np.mean(xc)) - cx) / cfg.focal_length_x)
                ) if len(xc) else 0.

                r   = scales
                csv_row = {
                    "img_id": index, "file_name": base_fn,
                    "mask_id": inst["local_idx"], "mask_uid": inst.get("uid",""),
                    "mask_score": score,
                    "super_cat": inst["super_cat"], "sub_cat": inst["sub_cat"],
                    "horizontal_angle": angle_deg,
                    **{f"unscaled_{k}": r["unscaled"][k]
                       for k in ["rmse","abs_rel","sq_rel","rmse_log",
                                  "delta1","delta2","delta3","avg_depth","min_depth"]},
                    "median_scale": r["median"]["scale"],
                    **{f"median_{k}": r["median"][k]
                       for k in ["rmse","abs_rel","sq_rel","rmse_log",
                                  "delta1","delta2","delta3","avg_depth","min_depth"]},
                    "mean_scale": r["mean"]["scale"],
                    **{f"mean_{k}": r["mean"][k]
                       for k in ["rmse","abs_rel","sq_rel","rmse_log",
                                  "delta1","delta2","delta3","avg_depth","min_depth"]},
                    "affine_scale": r["affine"]["scale"],
                    "affine_shift": r["affine"]["shift"],
                    **{f"affine_{k}": r["affine"][k]
                       for k in ["rmse","abs_rel","sq_rel","rmse_log",
                                  "delta1","delta2","delta3","avg_depth","min_depth"]},
                    "lidar_avg_depth": float(np.mean(lidar_v)),
                    "lidar_min_depth": float(np.min(lidar_v)),
                    "rgb_image_path":  rgb_out,
                    "gt_depth_map_path":  gt_out,
                    "da3_pred_map_path":  da3_out,
                    "mask_path": str(inst.get("path","")),
                }
                pd.DataFrame([csv_row]).to_csv(output_csv, mode="a",
                                                header=False, index=False)
        except Exception as exc:
            log.error(f"Error {base_fn}: {exc}", exc_info=False)

    log.info(f"Eval complete → {output_csv}")
    if skipped_no_masks:
        log.warning(f"  {skipped_no_masks} frames skipped (no masks found).")

## §12 · Plotting: PLOTTING

In [13]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale",
    "mean":     "Mean Scale",
    "affine":   "Affine Scale",
}


def _save_fig(fig, path: str, dpi: int = 150):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info(f"  ✓  {os.path.basename(path)}")


def _load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    drop_cols = [c for c in ["unscaled_rmse","median_rmse","mean_rmse","affine_rmse",
                               "lidar_avg_depth","lidar_min_depth"] if c in df.columns]
    df = df.dropna(subset=drop_cols)
    return df


def generate_analysis_plots(df: pd.DataFrame, plot_dir: str,
                              model_label: str, max_depth: float = 80.) -> None:
    """Generate 12 standard analysis plots for a single model's results."""
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    os.makedirs(plot_dir, exist_ok=True)
    tag = model_label

    # Plot 01: GT vs Predicted
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    variants = [("Unscaled","lidar_avg_depth","unscaled_avg_depth"),
                ("Median Scale","lidar_avg_depth","median_avg_depth"),
                ("Mean Scale","lidar_avg_depth","mean_avg_depth"),
                ("Affine Scale","lidar_avg_depth","affine_avg_depth")]
    for ax, (title, xc, yc), color in zip(axes.flatten(), variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x=xc, y=yc,
                    scatter_kws={"s":20,"alpha":0.3,"color":color},
                    line_kws={"color":"red","linewidth":2}, lowess=True)
        ax.plot([0,max_depth],[0,max_depth],"k--",linewidth=1.5,label="Ideal 1:1")
        ax.set_title(f"{tag}  |  {title}"); ax.set_xlabel("LiDAR GT (m)"); ax.set_ylabel("Pred (m)")
        ax.set_xlim(0,max_depth); ax.set_ylim(0,max_depth); ax.legend(fontsize=9)
    fig.suptitle(f"GT vs Predicted Depth  [{tag}]", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"01_gt_vs_pred.png"))

    # Plot 02: GT vs RMSE
    variants2 = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                  ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants2]].max().max() * 1.12
    fig, axes = plt.subplots(2,2,figsize=(16,14))
    for ax, (title, yc), color in zip(axes.flatten(), variants2, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=yc,
                    scatter_kws={"s":20,"alpha":0.3,"color":color},
                    line_kws={"color":"red","linewidth":2}, lowess=True)
        ax.set_title(f"{tag}  |  {title}"); ax.set_xlabel("LiDAR GT (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0,max_depth); ax.set_ylim(0,ymax)
    fig.suptitle(f"GT Depth vs RMSE  [{tag}]", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"02_gt_vs_rmse.png"))

    # Plot 03: Angle vs RMSE
    fig, axes = plt.subplots(2,2,figsize=(16,14))
    for ax, (title, yc), color in zip(axes.flatten(), variants2, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=yc,
                    scatter_kws={"s":20,"alpha":0.3,"color":color},
                    line_kws={"color":"red","linewidth":2}, lowess=True)
        ax.set_title(f"{tag}  |  {title}: Angle vs RMSE"); ax.set_xlabel("Horizontal Angle (°)")
        ax.set_ylabel("RMSE (m)"); ax.set_ylim(0,ymax)
    fig.suptitle(f"Horizontal Angle vs RMSE  [{tag}]", fontsize=14, y=1.01)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"03_angle_vs_rmse.png"))

    # Plot 04: Scale comparison bar chart
    m_specs = [("RMSE (m) ↓",["unscaled_rmse","median_rmse","mean_rmse","affine_rmse"]),
               ("Abs Rel ↓",["unscaled_abs_rel","median_abs_rel","mean_abs_rel","affine_abs_rel"]),
               ("δ<1.25 ↑",["unscaled_delta1","median_delta1","mean_delta1","affine_delta1"])]
    labels  = ["Unscaled","Median","Mean","Affine"]
    colors  = list(SCALE_COLORS.values())
    fig, axes = plt.subplots(1,3,figsize=(18,6))
    for ax, (mn, cols) in zip(axes, m_specs):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
        ax.set_title(mn); ax.set_ylabel(mn); ax.set_ylim(0,max(vals)*1.22); ax.grid(axis="y",alpha=0.35)
    fig.suptitle(f"Scale Method Comparison  [{tag}]", fontsize=14, y=1.02)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"04_scale_comparison_bars.png"))

    # Plot 05: Model vs LiDAR by category
    cats = sorted(df["super_cat"].dropna().unique())
    n    = len(cats)
    if n > 0:
        fig, axes = plt.subplots(1,max(n,1),figsize=(7*n,6))
        if n==1: axes=[axes]
        for ax, cat in zip(axes,cats):
            sub = df[df["super_cat"]==cat]
            for key in SCALE_COLORS:
                ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                           s=15,alpha=0.35,c=SCALE_COLORS[key],label=SCALE_LABELS[key])
            ax.plot([0,max_depth],[0,max_depth],"k--",linewidth=1.5,label="1:1")
            ax.set_xlim(0,max_depth); ax.set_ylim(0,max_depth)
            ax.set_xlabel("LiDAR GT (m)"); ax.set_ylabel("Pred (m)")
            ax.set_title(f"Category: {cat}"); ax.legend(fontsize=7)
        fig.suptitle(f"Pred vs LiDAR — by Category  [{tag}]", fontsize=13)
        plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"05_model_vs_lidar_by_category.png"))

    # Plot 06: RMSE distributions
    fig, axes = plt.subplots(2,2,figsize=(16,10))
    for ax, (lb, col), color in zip(axes.flatten(), variants2, SCALE_COLORS.values()):
        v = df[col].dropna()
        ax.hist(v,bins=60,color=color,alpha=0.72,edgecolor="white")
        ax.axvline(v.mean(),color="black",linestyle="--",linewidth=1.8,label=f"Mean={v.mean():.3f}")
        ax.axvline(v.median(),color="grey",linestyle=":",linewidth=1.8,label=f"Median={v.median():.3f}")
        ax.set_title(f"RMSE Dist — {lb}"); ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle(f"RMSE Distributions  [{tag}]", fontsize=14)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"06_rmse_distributions.png"))

    # Plot 07: Scale distributions
    sspec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
             ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
             ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1,3,figsize=(18,5))
    for ax, (col,lb,color) in zip(axes,sspec):
        v = df[col].dropna()
        ax.hist(v,bins=60,color=color,alpha=0.72,edgecolor="white")
        ax.axvline(1.0,color="black",linestyle="--",linewidth=1.8,label="Scale=1.0")
        ax.axvline(v.mean(),color="red",linestyle=":",linewidth=1.8,label=f"Mean={v.mean():.3f}")
        ax.set_title(f"{lb} Distribution"); ax.set_xlabel("Scale Factor"); ax.legend(fontsize=9)
    fig.suptitle(f"Scale Factor Distributions  [{tag}]", fontsize=14)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"07_scale_distributions.png"))

    # Plot 08: 3D error surfaces
    df_c  = df.dropna(subset=["lidar_avg_depth","horizontal_angle"]+[v[1] for v in variants2])
    x_min, x_max = df_c["lidar_avg_depth"].min(), df_c["lidar_avg_depth"].max()
    y_min, y_max = df_c["horizontal_angle"].min(), df_c["horizontal_angle"].max()
    fig   = plt.figure(figsize=(26,6))
    for sp, (label, zcol), color in zip(range(1,5), variants2,
                                         ["b","g","orange","purple"]):
        ax = fig.add_subplot(1,4,sp,projection="3d")
        ax.scatter(df_c["lidar_avg_depth"],df_c["horizontal_angle"],df_c[zcol],
                   c=color,alpha=0.3,s=6)
        pipe = make_pipeline(PolynomialFeatures(2),LinearRegression())
        Xf   = df_c[["lidar_avg_depth","horizontal_angle"]].values
        Zf   = df_c[zcol].values
        pipe.fit(Xf,Zf)
        xs,ys = np.meshgrid(np.linspace(x_min,x_max,30),np.linspace(y_min,y_max,30))
        zs = pipe.predict(np.c_[xs.ravel(),ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs,ys,zs,color=color,alpha=0.2)
        ax.set_xlabel("GT Depth (m)",fontsize=8); ax.set_ylabel("H.Angle(°)",fontsize=8)
        ax.set_zlabel("RMSE(m)",fontsize=8); ax.set_title(label,fontsize=11)
    fig.suptitle(f"3D Error Surfaces  [{tag}]", fontsize=13)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"08_3d_error_surfaces.png"))

    # Plot 09: Depth-bin analysis
    bins  = np.arange(0,max_depth+10,10)
    lbls  = [f"{int(b)}-{int(b+10)}m" for b in bins[:-1]]
    df2   = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"],bins=bins,labels=lbls)
    fig, axes = plt.subplots(1,2,figsize=(18,6))
    ax = axes[0]
    for key in SCALE_COLORS:
        agg = df2.groupby("depth_bin",observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index,agg.values,"o-",color=SCALE_COLORS[key],label=SCALE_LABELS[key],linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)"); ax.set_title("RMSE by Depth Bin")
    ax.legend(); ax.tick_params(axis="x",rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in SCALE_COLORS:
        bias = df2.groupby("depth_bin",observed=True)[f"{key}_avg_depth"].mean() \
             - df2.groupby("depth_bin",observed=True)["lidar_avg_depth"].mean()
        ax.plot(bias.index,bias.values,"o-",color=SCALE_COLORS[key],label=SCALE_LABELS[key],linewidth=2)
    ax.axhline(0,color="black",linestyle="--",linewidth=1.5)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Bias (m)"); ax.set_title("Depth Bias by Bin")
    ax.legend(); ax.tick_params(axis="x",rotation=45); ax.grid(alpha=0.3)
    fig.suptitle(f"Depth-Bin Analysis  [{tag}]", fontsize=14)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"09_depth_bin_analysis.png"))

    # Plot 10: RMSE by category
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) >= 2:
        keys = list(SCALE_COLORS.keys()); n_keys = len(keys)
        x, w = np.arange(len(cats)), 0.18
        fig, ax = plt.subplots(figsize=(13,6))
        for i, key in enumerate(keys):
            vals   = [df[df["super_cat"]==c][f"{key}_rmse"].mean() for c in cats]
            offset = (i-(n_keys-1)/2.) * w
            bars   = ax.bar(x+offset,vals,width=w,label=SCALE_LABELS[key],color=SCALE_COLORS[key])
            ax.bar_label(bars,fmt="%.2f",fontsize=8,padding=2)
        ax.set_xticks(x); ax.set_xticklabels(cats,fontsize=11)
        ax.set_ylabel("Mean RMSE (m)"); ax.set_title(f"RMSE per Category  [{tag}]")
        ax.legend(); ax.grid(axis="y",alpha=0.35)
        plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"10_rmse_by_category.png"))

    # Plot 11: Scale factor vs depth
    fig, axes = plt.subplots(1,3,figsize=(18,5))
    for ax, (col,lb,color) in zip(axes,sspec):
        sns.regplot(ax=ax,data=df,x="lidar_avg_depth",y=col,
                    scatter_kws={"s":15,"alpha":0.25,"color":color},
                    line_kws={"color":"red","linewidth":2},lowess=True)
        ax.axhline(1.0,color="black",linestyle="--",linewidth=1.5,label="Scale=1.0")
        ax.set_xlabel("LiDAR GT (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{lb} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle(f"Scale Factor vs GT Depth  [{tag}]", fontsize=14)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"11_scale_vs_depth.png"))

    # Plot 00: Summary table image
    rows_data, tspecs = [], [
        ("Unscaled",    "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None),
        ("Median Scale","median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None),
        ("Mean Scale",  "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None),
        ("Affine Scale","affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift"),
    ]
    for lb,rc,ac,dc,sc,shc in tspecs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        rows_data.append([lb, f"{sv:.4f}", f"{shv:.4f}" if shc else "—",
                          f"{df[rc].mean():.4f}", f"{df[ac].mean():.4f}",
                          f"{df[dc].mean():.4f}"])
    fig, ax = plt.subplots(figsize=(14,3)); ax.axis("off")
    tbl = ax.table(cellText=rows_data,
                   colLabels=["Scale Method","Avg Scale","Avg Shift",
                               "Avg RMSE (m)","Avg Abs Rel","Avg δ<1.25"],
                   cellLoc="center",loc="center",bbox=[0,0,1,1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2,1.9)
    for j in range(6):
        tbl[0,j].set_facecolor("#2c3e50"); tbl[0,j].set_text_props(color="white",fontweight="bold")
    for i in range(1,len(rows_data)+1):
        bg = "white" if i%2==0 else "#f0f4f8"
        for j in range(6): tbl[i,j].set_facecolor(bg)
    fig.suptitle(f"Summary Table  [{tag}]", fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"00_summary_table.png"), dpi=180)


def generate_comparison_plots(
    df_base: pd.DataFrame,
    df_ft:   pd.DataFrame,
    plot_dir: str,
    max_depth: float = 80.,
) -> None:
    """
    3 plots that directly compare baseline vs fine-tuned model performance.
    """
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    os.makedirs(plot_dir, exist_ok=True)

    # ── Comp 01: Side-by-side RMSE, Abs Rel, δ<1.25 bars ────────────────────
    m_spec = [("RMSE (m) ↓","_rmse"), ("Abs Rel ↓","_abs_rel"), ("δ<1.25 ↑","_delta1")]
    keys   = list(SCALE_COLORS.keys())
    x, w   = np.arange(len(keys)), 0.35
    fig, axes = plt.subplots(1,3,figsize=(18,6))
    for ax, (mn, suffix) in zip(axes, m_spec):
        b_vals = [df_base[f"{k}{suffix}"].mean() for k in keys]
        f_vals = [df_ft  [f"{k}{suffix}"].mean() for k in keys]
        bars_b = ax.bar(x - w/2, b_vals, w, label="Baseline", color="#3498db", alpha=0.85)
        bars_f = ax.bar(x + w/2, f_vals, w, label="Fine-tuned", color="#e67e22", alpha=0.85)
        ax.bar_label(bars_b, fmt="%.4f", padding=3, fontsize=8)
        ax.bar_label(bars_f, fmt="%.4f", padding=3, fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels([SCALE_LABELS[k] for k in keys], fontsize=10, rotation=15)
        ax.set_title(mn); ax.legend(); ax.grid(axis="y",alpha=0.3)
    fig.suptitle("Baseline vs Fine-tuned  —  Error Metric Comparison", fontsize=14, y=1.02)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"comp_01_metric_comparison.png"))

    # ── Comp 02: Improvement (%) bar chart ───────────────────────────────────
    def pct_improve(base_val, ft_val, higher_is_better=False):
        if abs(base_val) < 1e-9:
            return 0.
        if higher_is_better:
            return (ft_val - base_val) / abs(base_val) * 100.
        else:
            return (base_val - ft_val) / abs(base_val) * 100.

    improvements = {}
    for k in keys:
        improvements[SCALE_LABELS[k]] = {
            "RMSE":    pct_improve(df_base[f"{k}_rmse"].mean(),   df_ft[f"{k}_rmse"].mean()),
            "Abs Rel": pct_improve(df_base[f"{k}_abs_rel"].mean(),df_ft[f"{k}_abs_rel"].mean()),
            "δ<1.25":  pct_improve(df_base[f"{k}_delta1"].mean(), df_ft[f"{k}_delta1"].mean(), higher_is_better=True),
        }
    imp_df   = pd.DataFrame(improvements).T   # rows = scale methods, cols = metrics
    x_imp    = np.arange(len(imp_df))
    met_cols = imp_df.columns.tolist()
    w_imp    = 0.25
    mcolors  = ["#e74c3c","#27ae60","#8e44ad"]
    fig, ax  = plt.subplots(figsize=(14,6))
    for i,(mc,col) in enumerate(zip(met_cols,mcolors)):
        vals = imp_df[mc].values
        bars = ax.bar(x_imp + (i-1)*w_imp, vals, w_imp, label=mc, color=col, alpha=0.82)
        ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.axhline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_xticks(x_imp); ax.set_xticklabels(imp_df.index, fontsize=11)
    ax.set_ylabel("Improvement (%)"); ax.set_title("% Improvement: Fine-tuned vs Baseline")
    ax.legend(); ax.grid(axis="y",alpha=0.3)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"comp_02_improvement_pct.png"))

    # ── Comp 03: RMSE by depth bin — baseline vs finetuned overlay ───────────
    bins = np.arange(0,max_depth+10,10)
    lbls = [f"{int(b)}-{int(b+10)}m" for b in bins[:-1]]
    df_base2 = df_base.copy(); df_base2["depth_bin"] = pd.cut(df_base2["lidar_avg_depth"],bins=bins,labels=lbls)
    df_ft2   = df_ft.copy();   df_ft2["depth_bin"]   = pd.cut(df_ft2["lidar_avg_depth"],  bins=bins,labels=lbls)

    fig, axes = plt.subplots(2,2,figsize=(16,12))
    for ax, key in zip(axes.flatten(), keys):
        b_agg = df_base2.groupby("depth_bin",observed=True)[f"{key}_rmse"].mean()
        f_agg = df_ft2.groupby("depth_bin",  observed=True)[f"{key}_rmse"].mean()
        ax.plot(b_agg.index,b_agg.values,"o-",color="#3498db",label="Baseline",linewidth=2)
        ax.plot(f_agg.index,f_agg.values,"s-",color="#e67e22",label="Fine-tuned",linewidth=2)
        ax.fill_between(range(len(b_agg)),b_agg.values,f_agg.values,
                        where=f_agg.values<b_agg.values,alpha=0.15,color="green",label="Improvement")
        ax.fill_between(range(len(b_agg)),b_agg.values,f_agg.values,
                        where=f_agg.values>=b_agg.values,alpha=0.15,color="red",label="Regression")
        ax.set_title(f"RMSE by Depth — {SCALE_LABELS[key]}")
        ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
        ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=9); ax.grid(alpha=0.3)
    fig.suptitle("RMSE by Depth Bin: Baseline vs Fine-tuned", fontsize=14)
    plt.tight_layout(); _save_fig(fig, os.path.join(plot_dir,"comp_03_rmse_depth_bin.png"))

    log.info(f"Comparison plots saved to {plot_dir}")

## §13 · Summary Tables: SUMMARY TABLES

In [14]:
def print_summary(df: pd.DataFrame, label: str) -> None:
    W = 110
    print(f"\n{'═'*W}")
    print(f"  {label}  —  Evaluation Summary")
    print(f"  Masks: {len(df):,}   |   Frames: {df['img_id'].nunique():,}")
    print(f"{'═'*W}")
    hdr = (f"{'Scale Method':<24}{'Avg Scale':>11}{'Avg Shift':>11}"
           f"{'RMSE':>10}{'Abs Rel':>10}{'Sq Rel':>10}"
           f"{'RMSE-Log':>10}{'δ<1.25':>9}{'δ<1.5625':>10}{'δ<1.953':>10}"
           f"{'LiDAR avg':>11}")
    print(hdr); print("─"*W)
    specs = [
        ("Unscaled",    "unscaled_rmse","unscaled_abs_rel","unscaled_sq_rel","unscaled_rmse_log",
         "unscaled_delta1","unscaled_delta2","unscaled_delta3",None,None),
        ("Median Scale","median_rmse",  "median_abs_rel",  "median_sq_rel",  "median_rmse_log",
         "median_delta1",  "median_delta2",  "median_delta3",  "median_scale",None),
        ("Mean Scale",  "mean_rmse",    "mean_abs_rel",    "mean_sq_rel",    "mean_rmse_log",
         "mean_delta1",    "mean_delta2",    "mean_delta3",    "mean_scale",  None),
        ("Affine Scale","affine_rmse",  "affine_abs_rel",  "affine_sq_rel",  "affine_rmse_log",
         "affine_delta1",  "affine_delta2",  "affine_delta3",  "affine_scale","affine_shift"),
    ]
    for (lb,rc,ac,sqc,rlc,d1c,d2c,d3c,sc,shc) in specs:
        sv  = df[sc].mean()  if sc  and sc  in df.columns else 1.0
        shv = df[shc].mean() if shc and shc in df.columns else 0.0
        print(f"  {lb:<22}{sv:>11.4f}{shv:>11.4f}"
              f"{df[rc].mean():>10.4f}{df[ac].mean():>10.4f}"
              f"{df[sqc].mean():>10.4f}{df[rlc].mean():>10.4f}"
              f"{df[d1c].mean():>9.4f}{df[d2c].mean():>10.4f}{df[d3c].mean():>10.4f}"
              f"{df['lidar_avg_depth'].mean():>11.4f}")
    print("═"*W + "\n")


def save_summary_csv(df_base: pd.DataFrame, df_ft: pd.DataFrame,
                      out_path: str) -> None:
    """Save combined summary CSV comparing baseline and fine-tuned."""
    specs = [
        ("Unscaled",    "unscaled"),
        ("Median Scale","median"),
        ("Mean Scale",  "mean"),
        ("Affine Scale","affine"),
    ]
    rows = []
    for label, key in specs:
        for model_label, df in [("Baseline", df_base), ("Fine-tuned", df_ft)]:
            row = {"model": model_label, "scale_method": label}
            for metric in ["rmse","abs_rel","sq_rel","rmse_log","delta1","delta2","delta3"]:
                col = f"{key}_{metric}"
                row[metric] = df[col].mean() if col in df.columns else float("nan")
            sc_col = f"{key}_scale"
            sh_col = f"{key}_shift"
            row["avg_scale"] = df[sc_col].mean() if sc_col in df.columns else 1.0
            row["avg_shift"] = df[sh_col].mean() if sh_col in df.columns else 0.0
            rows.append(row)

    summary = pd.DataFrame(rows)
    summary.to_csv(out_path, index=False)
    log.info(f"Summary CSV saved: {out_path}")

## §14 · Entry-Point Functions: ENTRY POINT

In [15]:
def parse_args():
    p = argparse.ArgumentParser(
        description="DA3METRIC-LARGE: KITTI self-supervised fine-tuning + evaluation",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--phase", default="all",
                   choices=["all","train","eval_baseline","eval_finetuned","plots"],
                   help=("all = run all phases sequentially; "
                         "train = fine-tune only; "
                         "eval_baseline = evaluate pre-trained model only; "
                         "eval_finetuned = evaluate fine-tuned model only; "
                         "plots = generate plots from existing CSVs"))

    # Paths
    p.add_argument("--kitti_root",  default="/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset")
    p.add_argument("--mask_root",   default="/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output")
    p.add_argument("--output_dir",  default="/STORAGE/Arka/a_MDE_Code/da3_kitti_finetuned")
    p.add_argument("--eval_dir",    default="/STORAGE/Arka/a_MDE_Code/da3_kitti_eval")
    p.add_argument("--cache_dir",   default="/STORAGE/Arka/cache")

    # Training
    p.add_argument("--epochs",        type=int,   default=30)
    p.add_argument("--lr",            type=float, default=1e-5)
    p.add_argument("--grad_accum",    type=int,   default=8)
    p.add_argument("--save_every",    type=int,   default=5)
    p.add_argument("--min_mask_pixels", type=int, default=200)
    p.add_argument("--resume",        action="store_true")
    p.add_argument("--no_amp",        action="store_true")
    p.add_argument("--no_8bit_adam",  action="store_true")

    # Loss weights
    p.add_argument("--lambda_variance",   type=float, default=1.0)
    p.add_argument("--lambda_refinement", type=float, default=0.5)
    p.add_argument("--lambda_ratio",      type=float, default=0.3)
    p.add_argument("--lambda_smooth",     type=float, default=0.05)
    p.add_argument("--lambda_kd",         type=float, default=0.2)

    # Evaluation
    p.add_argument("--use_train_for_eval", action="store_true")
    p.add_argument("--no_save_npy",        action="store_true")

    return p.parse_args()


def build_config(args) -> Config:
    return Config(
        kitti_root       = args.kitti_root,
        mask_root        = args.mask_root,
        output_dir       = args.output_dir,
        eval_dir         = args.eval_dir,
        cache_dir        = args.cache_dir,
        epochs           = args.epochs,
        lr               = args.lr,
        grad_accum       = args.grad_accum,
        save_every       = args.save_every,
        min_mask_pixels  = args.min_mask_pixels,
        resume           = args.resume,
        mixed_precision  = not args.no_amp,
        use_8bit_adam    = not args.no_8bit_adam,
        lambda_variance  = args.lambda_variance,
        lambda_refinement= args.lambda_refinement,
        lambda_ratio     = args.lambda_ratio,
        lambda_smooth    = args.lambda_smooth,
        lambda_kd        = args.lambda_kd,
        use_train_for_eval = args.use_train_for_eval,
        save_npy         = not args.no_save_npy,
    )


def phase_train(cfg: Config):
    log.info("\n" + "═"*60)
    log.info("  PHASE 1 — SELF-SUPERVISED FINE-TUNING")
    log.info("═"*60)
    train(cfg)


def phase_eval(cfg: Config, finetuned: bool) -> str:
    tag   = "finetuned" if finetuned else "baseline"
    label = "Fine-tuned" if finetuned else "Baseline"
    log.info(f"\n{'═'*60}")
    log.info(f"  PHASE — EVALUATION  [{label}]")
    log.info("═"*60)

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model   = load_da3_model(cfg.model_id, cfg.cache_dir, device, train=False)

    if finetuned:
        ok = load_best_checkpoint(model, cfg)
        if not ok:
            log.error("No fine-tuned checkpoint found. Run --phase train first.")
            return ""

    out_dir = os.path.join(cfg.eval_dir, tag)
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"da3_metric_{tag}_results.csv")

    val_list = index_kitti(cfg)
    model.eval()
    process_dataset_eval(model, val_list, cfg, csv_path, out_dir, label=label)
    return csv_path


def phase_plots(cfg: Config,
                csv_base: str = "",
                csv_ft:   str = "") -> None:
    log.info("\n" + "═"*60)
    log.info("  PHASE — PLOTS & SUMMARY")
    log.info("═"*60)

    # Auto-detect CSV paths if not provided
    if not csv_base:
        csv_base = os.path.join(cfg.eval_dir, "baseline",
                                "da3_metric_baseline_results.csv")
    if not csv_ft:
        csv_ft   = os.path.join(cfg.eval_dir, "finetuned",
                                "da3_metric_finetuned_results.csv")

    have_base = os.path.exists(csv_base)
    have_ft   = os.path.exists(csv_ft)

    if not have_base and not have_ft:
        log.error("No result CSVs found. Run eval phases first.")
        return

    if have_base:
        df_base = _load_and_clean(csv_base)
        print_summary(df_base, "Baseline DA3METRIC-LARGE")
        plot_dir_base = os.path.join(cfg.eval_dir, "baseline", "plots")
        generate_analysis_plots(df_base, plot_dir_base, "Baseline", cfg.max_depth)

    if have_ft:
        df_ft = _load_and_clean(csv_ft)
        print_summary(df_ft, "Fine-tuned DA3METRIC-LARGE")
        plot_dir_ft = os.path.join(cfg.eval_dir, "finetuned", "plots")
        generate_analysis_plots(df_ft, plot_dir_ft, "Fine-tuned", cfg.max_depth)

    if have_base and have_ft:
        comp_dir = os.path.join(cfg.eval_dir, "comparison_plots")
        generate_comparison_plots(df_base, df_ft, comp_dir, cfg.max_depth)

        summary_csv = os.path.join(cfg.eval_dir, "combined_summary.csv")
        save_summary_csv(df_base, df_ft, summary_csv)

        # Final combined console table
        W = 115
        print(f"\n{'═'*W}")
        print("  COMBINED SUMMARY: Baseline vs Fine-tuned DA3METRIC-LARGE")
        print(f"{'═'*W}")
        print(f"  {'Scale Method':<22}  {'Model':<12}  "
              f"{'RMSE':>9}  {'Abs Rel':>9}  {'δ<1.25':>8}  {'Δ RMSE':>9}  {'Δ δ<1.25':>10}")
        print("─"*W)
        for key, label in [("unscaled","Unscaled"),("median","Median Scale"),
                             ("mean","Mean Scale"),("affine","Affine Scale")]:
            b_rmse  = df_base[f"{key}_rmse"].mean()
            f_rmse  = df_ft  [f"{key}_rmse"].mean()
            b_abs   = df_base[f"{key}_abs_rel"].mean()
            f_abs   = df_ft  [f"{key}_abs_rel"].mean()
            b_d1    = df_base[f"{key}_delta1"].mean()
            f_d1    = df_ft  [f"{key}_delta1"].mean()
            print(f"  {label:<22}  {'Baseline':<12}  "
                  f"{b_rmse:>9.4f}  {b_abs:>9.4f}  {b_d1:>8.4f}  {'—':>9}  {'—':>10}")
            arrow = "↓" if f_rmse < b_rmse else "↑"
            print(f"  {'':22}  {'Fine-tuned':<12}  "
                  f"{f_rmse:>9.4f}  {f_abs:>9.4f}  {f_d1:>8.4f}  "
                  f"{(f_rmse-b_rmse):>+9.4f}  {(f_d1-b_d1):>+10.4f} {arrow}")
            print(f"  {'─'*W}")
        print()


def main():
    args = parse_args()
    cfg  = build_config(args)

    log.info("═"*65)
    log.info("  DA3METRIC-LARGE — KITTI Fine-tuning & Evaluation Pipeline")
    log.info("═"*65)
    log.info(f"  Phase     : {args.phase}")
    log.info(f"  KITTI root: {cfg.kitti_root}")
    log.info(f"  Mask root : {cfg.mask_root}")
    log.info(f"  Output    : {cfg.output_dir}")
    log.info(f"  Eval dir  : {cfg.eval_dir}")
    log.info("")

    csv_base, csv_ft = "", ""

    if args.phase in ("all", "train"):
        phase_train(cfg)

    if args.phase in ("all", "eval_baseline"):
        csv_base = phase_eval(cfg, finetuned=False)

    if args.phase in ("all", "eval_finetuned"):
        csv_ft = phase_eval(cfg, finetuned=True)

    if args.phase in ("all", "plots"):
        phase_plots(cfg, csv_base, csv_ft)

    if args.phase == "eval_baseline":
        if csv_base:
            df = _load_and_clean(csv_base)
            print_summary(df, "Baseline DA3METRIC-LARGE")

    if args.phase == "eval_finetuned":
        if csv_ft:
            df = _load_and_clean(csv_ft)
            print_summary(df, "Fine-tuned DA3METRIC-LARGE")


# if __name__ == "__main__":
#     main()

## 🚀 Pipeline Execution

Configure paths and phase below, then run **Run All** (or execute cells individually).

| `PHASE` value | What runs |
|---|---|
| `"all"` | Fine-tune → Baseline eval → Fine-tuned eval → Plots |
| `"train"` | Fine-tuning only |
| `"eval_baseline"` | Pre-trained model evaluation only |
| `"eval_finetuned"` | Fine-tuned model evaluation only |
| `"plots"` | Re-generate plots from existing CSVs |


In [19]:
# ── Edit paths / hyperparameters here ─────────────────────────────────────
PHASE = "plots"          # all | train | eval_baseline | eval_finetuned | plots
RESUME = True          # Resume fine-tuning from latest checkpoint if available

cfg = Config(
    # ── Paths ────────────────────────────────────────────────────────────
    kitti_root  = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_small/",
    mask_root   = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output",
    output_dir  = "/STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_finetuned",
    eval_dir    = "/STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_eval",
    cache_dir   = "/STORAGE/Arka/cache",

    # ── Training ─────────────────────────────────────────────────────────
    epochs          = 20,
    lr              = 1e-5,
    grad_accum      = 8,
    save_every      = 5,
    min_mask_pixels = 200,
    resume          = RESUME,
    mixed_precision = True,
    use_8bit_adam   = True,

    # ── Loss weights ─────────────────────────────────────────────────────
    lambda_variance   = 1.0,   # Var(Δ depth) — uniformity of change
    lambda_refinement = 0.5,   # MSE(Δ, mean Δ) — refinement
    lambda_ratio      = 0.3,   # Var(d_{t+1}/d_t) — scale-invariant
    lambda_smooth     = 0.05,  # Edge-aware smoothness
    lambda_kd         = 0.2,   # Self-distillation (frozen teacher)

    # ── Evaluation ────────────────────────────────────────────────────────
    use_train_for_eval = False,
    save_npy           = True,
)

# ── Run selected phase ─────────────────────────────────────────────────────
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_pipeline")

csv_base, csv_ft = "", ""

if PHASE in ("all", "train"):
    phase_train(cfg)

if PHASE in ("all", "eval_baseline"):
    csv_base = phase_eval(cfg, finetuned=False)

if PHASE in ("all", "eval_finetuned"):
    csv_ft = phase_eval(cfg, finetuned=True)

if PHASE in ("all", "plots"):
    phase_plots(cfg, csv_base, csv_ft)

# Stand-alone summary if running just one eval phase
if PHASE == "eval_baseline" and csv_base:
    df = _load_and_clean(csv_base)
    print_summary(df, "Baseline DA3METRIC-LARGE")

if PHASE == "eval_finetuned" and csv_ft:
    df = _load_and_clean(csv_ft)
    print_summary(df, "Fine-tuned DA3METRIC-LARGE")

11:03:54  INFO      
════════════════════════════════════════════════════════════
11:03:54  INFO        PHASE — PLOTS & SUMMARY
11:03:54  INFO      ════════════════════════════════════════════════════════════



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Baseline DA3METRIC-LARGE  —  Evaluation Summary
  Masks: 412   |   Frames: 23
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method              Avg Scale  Avg Shift      RMSE   Abs Rel    Sq Rel  RMSE-Log   δ<1.25  δ<1.5625   δ<1.953  LiDAR avg
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                   1.0000     0.0000    7.2203    0.1845    4.2416    0.1888   0.8554    0.9170    0.9619    32.4943
  Median Scale               0.9958     0.0000    4.1226    0.0729    1.2464    0.1176   0.9296    0.9686    0.9849    32.4943
  Mean Scale                 0.9701     0.0000    3.6458    0.0816    0.8288    0.1117   0.9130    0.9737    0.9902    32.4943
  Affine Scale               0.2778    20.9674    1.3613    0.0282    0.1887  

11:03:55  INFO        ✓  01_gt_vs_pred.png
11:03:56  INFO        ✓  02_gt_vs_rmse.png
11:03:57  INFO        ✓  03_angle_vs_rmse.png
11:03:58  INFO        ✓  04_scale_comparison_bars.png
11:03:59  INFO        ✓  05_model_vs_lidar_by_category.png
11:04:00  INFO        ✓  06_rmse_distributions.png
11:04:00  INFO        ✓  07_scale_distributions.png
11:04:02  INFO        ✓  08_3d_error_surfaces.png
11:04:02  INFO        ✓  09_depth_bin_analysis.png
11:04:02  INFO        ✓  10_rmse_by_category.png
11:04:03  INFO        ✓  11_scale_vs_depth.png
11:04:04  INFO        ✓  00_summary_table.png



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Fine-tuned DA3METRIC-LARGE  —  Evaluation Summary
  Masks: 412   |   Frames: 23
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method              Avg Scale  Avg Shift      RMSE   Abs Rel    Sq Rel  RMSE-Log   δ<1.25  δ<1.5625   δ<1.953  LiDAR avg
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                   1.0000     0.0000    8.8264    0.2260    2.9774    0.2629   0.5504    0.8372    0.9772    32.4943
  Median Scale               1.1666     0.0000    2.3350    0.0453    0.3715    0.0723   0.9680    0.9876    0.9949    32.4943
  Mean Scale                 1.1701     0.0000    2.1853    0.0501    0.3177    0.0688   0.9647    0.9926    0.9971    32.4943
  Affine Scale               0.9760    -2.3191    1.3239    0.0273    0.1545

11:04:05  INFO        ✓  01_gt_vs_pred.png
11:04:06  INFO        ✓  02_gt_vs_rmse.png
11:04:07  INFO        ✓  03_angle_vs_rmse.png
11:04:07  INFO        ✓  04_scale_comparison_bars.png
11:04:08  INFO        ✓  05_model_vs_lidar_by_category.png
11:04:09  INFO        ✓  06_rmse_distributions.png
11:04:10  INFO        ✓  07_scale_distributions.png
11:04:11  INFO        ✓  08_3d_error_surfaces.png
11:04:12  INFO        ✓  09_depth_bin_analysis.png
11:04:12  INFO        ✓  10_rmse_by_category.png
11:04:13  INFO        ✓  11_scale_vs_depth.png
11:04:13  INFO        ✓  00_summary_table.png
11:04:13  INFO        ✓  comp_01_metric_comparison.png
11:04:14  INFO        ✓  comp_02_improvement_pct.png
11:04:14  INFO        ✓  comp_03_rmse_depth_bin.png
11:04:14  INFO      Comparison plots saved to /STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_eval/comparison_plots
11:04:15  INFO      Summary CSV saved: /STORAGE/Arka/a_MDE_Code/4_Self_supervised_FT/da3_kitti_eval/combined_summary.csv



═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  COMBINED SUMMARY: Baseline vs Fine-tuned DA3METRIC-LARGE
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Scale Method            Model              RMSE    Abs Rel    δ<1.25     Δ RMSE    Δ δ<1.25
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                Baseline         7.2203     0.1845    0.8554          —           —
                          Fine-tuned       8.8264     0.2260    0.5504    +1.6061     -0.3050 ↑
  ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Median Scale            Baseline         4.1226     0.0729    0.9296          —           —
                          Fine-tuned       2.3350     0.0453    0.9680    -1.7876     +0.0384 ↓
